# Factor Zoo

Compare all available price-based factors on synthetic data.

In [ ]:
import pandas as pd
from factor_forge.backtest.engine import BacktestEngine
from factor_forge.data.loader import generate_synthetic_prices
from factor_forge.factors.base import FactorCategory
from factor_forge.factors.registry import FactorRegistry

prices = generate_synthetic_prices(["A", "B", "C", "D", "E"], "2015-01-01", "2024-01-01")
registry = FactorRegistry()
registry.load_builtin()

rows = []
for name in registry.list_factors():
    factor = registry.get(name)
    if not set(factor.inputs) <= set(prices.columns):
        continue
    engine = BacktestEngine(factor=factor, prices=prices, transaction_cost_bps=10.0)
    result = engine.run()
    rows.append({"factor": name, **result.metrics})

summary = pd.DataFrame(rows).sort_values("sharpe_ratio", ascending=False)
summary